In [1]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

!pip install -q langchain langchain-community langchain-text-splitters -U langchain-core sentence-transformers faiss-cpu pymupdf beautifulsoup4 requests==2.32.4 urllib3 tqdm vaderSentiment textblob


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.0/625.0 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.

In [2]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

import os
import json
import time
import requests
import fitz  # PyMuPDF
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import urllib3
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

RAG_DIR = '/content/drive/MyDrive/FranchiseOps_AI/rag_documents'
CUSTOM_PDF_DIR = '/content/drive/MyDrive/FranchiseOps_AI/custom_pdfs'
os.makedirs(RAG_DIR, exist_ok=True)
os.makedirs(CUSTOM_PDF_DIR, exist_ok=True)
print(f'✅ Output Directory Ready: {RAG_DIR}')
print(f'✅ Custom PDF Drop Folder Ready: {CUSTOM_PDF_DIR}')
print('   (Drop your own PDFs into this Drive folder any time before running the ingestion cell below.)')


Mounted at /content/drive
✅ Output Directory Ready: /content/drive/MyDrive/FranchiseOps_AI/rag_documents
✅ Custom PDF Drop Folder Ready: /content/drive/MyDrive/FranchiseOps_AI/custom_pdfs
   (Drop your own PDFs into this Drive folder any time before running the ingestion cell below.)


In [3]:
HTML_SOURCES = [
    # Marketing & Consumer Research
    "https://www.marketingweek.com",
    "https://hbr.org/topic/subject/marketing",
    "https://www.nielsen.com/insights",
    "https://www.mckinsey.com/capabilities/growth-marketing-and-sales/our-insights",
    "https://www.thinkwithgoogle.com",
    "https://www.campaignlive.co.uk",
    "https://www.warc.com/newsandopinion/opinion",

    # Customer Experience
    "https://www.pwc.com/us/en/services/consulting/library/consumer-intelligence-series.html",
    "https://www.zendesk.com/blog/customer-experience",
    "https://www.salesforce.com/resources/articles/customer-experience",

    # HR & Attrition Research
    "https://www.shrm.org/topics-tools/tools/how-to-guides/how-to-conduct-stay-interviews",
    "https://www.gallup.com/workplace/247391/fixable-problem-costs-businesses-trillion.aspx",
    "https://hbr.org/topic/subject/hr-management",
    "https://www.mckinsey.com/capabilities/people-and-organizational-performance/our-insights",

    # Food Safety & FSSAI
    "https://www.fssai.gov.in",
    "https://www.fssai.gov.in/cms/food-safety-and-standards-act-2006.php",
    "https://www.fssai.gov.in/cms/rules.php",
    "https://www.fssai.gov.in/cms/regulations.php",
    "https://www.fssai.gov.in/cms/gazette-notifications.php",
    "https://www.fssai.gov.in/cms/licensing.php",

    # Labour Laws
    "https://labour.gov.in/minimum-wages-act",
    "https://labour.gov.in/payment-of-wages-act",
    "https://labour.gov.in/maternity-benefit-act",
    "https://labour.gov.in/child-labour",
    "https://labour.gov.in/factories-act",
    "https://labour.gov.in/employees-provident-fund-organisation",
    "https://labour.gov.in/employees-state-insurance-corporation",
    "https://labour.gov.in/occupational-safety-and-health",
    "https://labour.gov.in/social-security",
    "https://labour.gov.in/industrial-relations",
    "https://labour.gov.in/bonus-act",
    "https://labour.gov.in/gratuity-act",

    # OSHA
    "https://www.osha.gov/workers",
    "https://www.osha.gov/employers",
    "https://www.osha.gov/laws-regs",
    "https://www.osha.gov/heat-exposure",
    "https://www.osha.gov/young-workers",
    "https://www.osha.gov/ergonomics",
    "https://www.osha.gov/personal-protective-equipment",

    # FDA
    "https://www.fda.gov/food/guidance-regulation-food-and-dietary-supplements",
    "https://www.fda.gov/food/buy-store-serve-safe-food",
    "https://www.fda.gov/food/new-era-smarter-food-safety",
    "https://www.fda.gov/food/food-labeling-nutrition",

    # WHO & International
    "https://www.who.int/news-room/fact-sheets/detail/food-safety",
    "https://www.who.int/health-topics/food-safety",
    "https://www.codexalimentarius.org",
    "https://www.fao.org/food-safety/en",
    "https://efsa.europa.eu/en/topics/topic/food-safety",
    "https://www.epfindia.gov.in",
    "https://www.esic.gov.in",
    "https://www.bis.gov.in",
    "https://apeda.gov.in",
    "https://www.mofpi.gov.in",
    "https://niti.gov.in",
    "https://www.startupindia.gov.in",
    "https://www.msme.gov.in",
    "https://mca.gov.in",
    "https://consumerhelpline.gov.in",
    "https://www.ncdrc.nic.in",
    "https://www.ilo.org/global/topics/safety-and-health-at-work/lang--en/index.htm",
    "https://www.ilo.org/global/topics/wages/minimum-wages/lang--en/index.htm",
    "https://www.shrm.org/topics-tools/tools/how-to-guides/how-to-develop-employee-handbook",
    "https://www.foodsafety.gov",
    "https://www.food.gov.uk",
    "https://www.food.gov.uk/business-guidance",
    "https://www.foodstandards.gov.au",
    "https://www.canada.ca/en/health-canada/services/food-nutrition.html",
    "https://www.sqfi.com",
    "https://www.brcgs.com/our-standards/food-safety",
    "https://www.mygfsi.com",
    "https://www.iso.org/committee/47638.html",
    "https://www.fao.org/nutrition/en",
    "https://www.wcfc.co",
    "https://www.bfa.org.uk",
    "https://www.ftc.gov/tips-advice/business-center/guidance/franchise-rule",
    "https://www.sba.gov/business-guide/launch-your-business/buy-franchise",
    "https://www.ifa.com",
    "https://www.qsrmagazine.com",
    "https://www.nrn.com",
    "https://www.restaurant.org/research-and-media/research/research-reports/state-of-the-industry",
    "https://www.indianspices.com",
    "https://www.coffeeboard.gov.in",
    "https://www.teaboard.gov.in",
    "https://mpeda.gov.in",
    "https://www.india.gov.in/spotlight/food-processing",
    "https://www.who.int/teams/nutrition-and-food-safety/food-safety",
    # Franchise Law & Disclosure
    "https://www.ftc.gov/business-guidance/industry/franchises-business-opportunities",
    "https://www.franchise.org/franchise-information",
    "https://www.franchisedirect.com/information/",
    "https://www.entrepreneur.com/franchise",
    "https://www.franchising.com",

    # Restaurant & QSR Operations
    "https://www.qsrmagazine.com/menu-innovations",
    "https://www.fsrmagazine.com",
    "https://www.restaurantbusinessonline.com",
    "https://www.nrn.com/operations",
    "https://www.foodservicedirector.com",

    # Quality & Food Safety Standards
    "https://www.iso.org/iso-22000-food-safety-management.html",
    "https://www.brcgs.com/",
    "https://www.mygfsi.com/how-to-implement/",
    "https://www.haccpalliance.org",
    "https://www.sqfi.com/sqf-food-safety-code/",

    # India-specific regulatory & tax
    "https://www.gst.gov.in",
    "https://www.incometax.gov.in",
    "https://www.mca.gov.in/content/mca/global/en/home.html",
    "https://www.indiafilings.com/learn/fssai-license/",
    "https://www.startupindia.gov.in/content/sih/en/government-schemes.html",

    # Workplace safety / POSH / HR compliance
    "https://wcd.nic.in/act/223",
    "https://www.shrm.org/topics-tools/topics/workplace-safety",
    "https://www.osha.gov/restaurants",
    "https://www.cdc.gov/niosh/topics/foodservice/default.html",
    "https://www.gallup.com/workplace/topic/employee-engagement.aspx",

    # Customer & Marketing analytics
    "https://www.qualtrics.com/experience-management/customer/",
    "https://www.forrester.com/customer-experience/",
    "https://www.statista.com/markets/13/food-beverage-tobacco/",
    "https://www.emarketer.com/topics/topic/restaurants",
    "https://www.euromonitor.com/consumer-foodservice",
]

PDF_SOURCES = [
    # --- FSSAI core acts, rules & regulations ---
    "https://www.fssai.gov.in/upload/uploadfiles/files/Food_Safety_and_Standards_Act_2006.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Licensing_and_Registration_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Packaging_and_Labelling_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Food_Product_Standards_and_Food_Additives_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Prohibition_and_Restriction_on_Sales_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Contaminants_Toxins_and_Residues_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Laboratory_and_Sample_Analysis_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Food_Safety_and_Standards_Rules_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Compendium_Food_Safety_Standards_Act.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Document_Organic_Foods.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Note_Nutraceuticals.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Manual_Milk_and_Milk_Products.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Manual_Meat_and_Meat_Products.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Manual_Fruits_and_Vegetables.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Document_Health_Supplements.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Note_Food_Additives.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Document_Edible_Oils.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/FSS_Organic_Foods_Regulations_2017.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Draft_FSS_Food_Recall_Procedure_Regulations_2017.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/FOSTAC_Training_Module_Basic.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/FOSTAC_Training_Module_Special.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Hygiene_Rating_Scheme_Guidelines.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Clean_Street_Food_Hub_Guidelines.pdf",

    # --- WHO ---
    "https://apps.who.int/iris/bitstream/handle/10665/44633/9789241501651_eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/255027/9789241512442-eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/43038/9241546123_eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/42913/9241546468.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/326765/9789240004467-eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/344474/9789240030060-eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/274671/9789241514217-eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/204348/9789241510066_eng.pdf",

    # --- FAO ---
    "https://www.fao.org/3/a0512e/a0512e00.pdf",
    "https://www.fao.org/3/i3794e/i3794e.pdf",
    "https://www.fao.org/3/y1579e/y1579e00.pdf",
    "https://www.fao.org/3/w9raw-e.pdf",
    "https://www.fao.org/3/y1390e/y1390e00.pdf",
    "https://www.fao.org/3/a-i4955e.pdf",
    "https://www.fao.org/3/cb4474en/cb4474en.pdf",
    "https://www.fao.org/3/ca5399en/ca5399en.pdf",
    "https://www.fao.org/3/i0142e/i0142e.pdf",
    "https://www.fao.org/3/y4893e/y4893e.pdf",
    "https://www.fao.org/3/ca0640en/CA0640EN.pdf",
    "https://www.fao.org/3/i9933en/i9933en.pdf",
    "https://www.fao.org/3/cc0461en/cc0461en.pdf",
    "https://www.fao.org/3/cb7408en/cb7408en.pdf",

    # --- Codex Alimentarius ---
    "https://www.fao.org/fao-who-codexalimentarius/sh-proxy/en/?lnk=1&url=https%3A%2F%2Fworkspace.fao.org%2Fsites%2Fcodex%2FStandards%2FCXP+1-1969%2FCXP001e.pdf",
    "https://www.fao.org/fao-who-codexalimentarius/sh-proxy/en/?lnk=1&url=https%3A%2F%2Fworkspace.fao.org%2Fsites%2Fcodex%2FStandards%2FCXS+1-1985%2FCXS001e.pdf",
    "https://www.fao.org/fao-who-codexalimentarius/sh-proxy/en/?lnk=1&url=https%3A%2F%2Fworkspace.fao.org%2Fsites%2Fcodex%2FStandards%2FCXS+193-1995%2FCXS193e.pdf",

    # --- ILO ---
    "https://www.ilo.org/wcmsp5/groups/public/---ed_norm/---normes/documents/publication/wcms_087817.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---dgreports/---dcomm/documents/publication/wcms_067588.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_protect/---protrav/---travail/documents/publication/wcms_712957.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_emp/---emp_ent/documents/publication/wcms_093580.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---asia/---ro-bangkok/---sro-new_delhi/documents/publication/wcms_631470.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---asia/---ro-bangkok/---sro-new_delhi/documents/publication/wcms_766448.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_protect/---protrav/---safework/documents/instructionalmaterial/wcms_113522.pdf",

    # --- OSHA ---
    "https://www.osha.gov/sites/default/files/publications/osha3165.pdf",
    "https://www.osha.gov/sites/default/files/publications/3148-06R-2011-English.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3151.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha2254.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3170.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3071.pdf",
    "https://www.osha.gov/sites/default/files/publications/OSHA_FS-3696.pdf",
    "https://www.osha.gov/sites/default/files/publications/OSHA3604.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3180.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha_3590.pdf",

    # --- FTC / SBA / Franchise Law (US) ---
    "https://www.ftc.gov/sites/default/files/documents/plain-language/bus70-franchise-rule.pdf",
    "https://www.sba.gov/sites/default/files/2022-08/Franchise-Guide.pdf",

    # --- World Bank ---
    "https://openknowledge.worldbank.org/bitstream/handle/10986/36613/9781464816109.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/35016/9781464816123.pdf",

    # --- India Labour Law Acts ---
    "https://labour.gov.in/sites/default/files/THE_MINIMUM_WAGES_ACT_1948.pdf",
    "https://labour.gov.in/sites/default/files/PaymentofWagesAct1936_0.pdf",
    "https://labour.gov.in/sites/default/files/TheMaternityBenefitAct_1961.pdf",
    "https://labour.gov.in/sites/default/files/payment_of_gratuity_act.pdf",
    "https://labour.gov.in/sites/default/files/ThePaymentofBonusAct1965.pdf",
    "https://consumeronline.gov.in/documents/ConsumerProtection-Act-2019.pdf",
    "https://www.epfindia.gov.in/site_docs/PDFs/Circulars/Y2022-23/Circular_EPFO_0111_2022.pdf",

    # --- Industry / management research reports ---
    "https://www.mckinsey.com/~/media/McKinsey/Business%20Functions/People%20and%20Organizational%20Performance/Our%20Insights/Reinventing%20the%20organization/Reinventing-the-organization.pdf",
    "https://www.foodstandards.gov.au/sites/default/files/documents/Meat%20Industry%20Operational%20Audit%20Guide.pdf",
    "https://www.ifi.unc.edu/wp-content/uploads/sites/863/2019/12/IFI-Report-Food-safety-culture.pdf",
    "https://efsa.onlinelibrary.wiley.com/doi/epdf/10.2903/j.efsa.2020.6098",
    "https://www.shrm.org/hr-today/trends-and-forecasting/research-and-surveys/Documents/SHRM%20Employee%20Job%20Satisfaction%20and%20Engagement.pdf",
    "https://www.gallup.com/workplace/349484/state-of-the-global-workplace-2022-report.aspx",
    "https://www.restaurant.org/downloads/pdfs/research/whats_hot_2023.pdf",

    # --- India govt annual reports / policy ---
    "https://bis.gov.in/wp-content/uploads/2022/02/Annual-Report-2020-21.pdf",
    "https://apeda.gov.in/apedawebsite/ANNUAL_REPORT/APEDA-Annual-Report-2021-22.pdf",
    "https://www.msme.gov.in/sites/default/files/MSME-Annual-Report-2021-22.pdf",
    "https://niti.gov.in/sites/default/files/2022-12/Food-Processing-Report.pdf",
    "https://www.mofpi.gov.in/sites/default/files/annual_report_2020-21.pdf",

    # --- Franchise & retail operations research ---
    "https://www.franchise.org/sites/default/files/2021-02/Franchise%20Business%20Economic%20Outlook%202021.pdf",
    "https://www.ftc.gov/system/files/documents/plain-language/bus70-franchise-rule-compliance-guide.pdf",
    "https://www.sba.gov/sites/default/files/2019-08/Franchise%20Disclosure%20Document%20Guide.pdf",
    "https://www.entrepreneur.com/downloads/franchise-500-methodology.pdf",

    # --- Food safety management systems (HACCP / ISO 22000 / BRC / SQF) ---
    "https://www.fda.gov/media/91262/download",
    "https://www.fda.gov/media/128301/download",
    "https://www.cdc.gov/foodsafety/pdfs/Retail-Food-Risk-Factor-Study-2018-Report.pdf",
    "https://www.iso.org/files/live/sites/isoorg/files/store/en/PUB100310.pdf",
    "https://www.food.gov.uk/sites/default/files/media/document/haccp-guide.pdf",
    "https://www.foodstandards.gov.au/publications/Documents/Food%20Safety%20Standards.pdf",

    # --- Occupational health / restaurant-specific safety ---
    "https://www.cdc.gov/niosh/docs/2014-102/pdfs/2014-102.pdf",
    "https://www.osha.gov/sites/default/files/publications/restaurant_hazards.pdf",
    "https://www.dol.gov/sites/dolgov/files/WHD/legacy/files/handy_reference_guide.pdf",

    # --- HR / attrition / employee engagement ---
    "https://www.shrm.org/hr-today/trends-and-forecasting/research-and-surveys/Documents/2021-Talent-Trends-Report.pdf",
    "https://www.gallup.com/workplace/321725/employee-engagement-technical-report.pdf",
    "https://www.mercer.com/content/dam/mercer/attachments/global/Talent/gl-2021-mercer-turnover-report.pdf",

    # --- Consumer protection / GST / tax reference (India) ---
    "https://cbic-gst.gov.in/pdf/CGST-Act-Updated-30092020.pdf",
    "https://www.incometax.gov.in/iec/foportal/sites/default/files/2022-09/Income-tax-Act-1961.pdf",
    "https://www.mca.gov.in/Ministry/pdf/CompaniesAct2013.pdf",

    # --- General restaurant / QSR benchmarking studies ---
    "https://www.restaurant.org/downloads/pdfs/research/2023-state-of-the-industry.pdf",
    "https://www.deloitte.com/content/dam/Deloitte/us/Documents/consumer-business/us-restaurant-of-the-future.pdf",
]

print(f"HTML Sources: {len(HTML_SOURCES)}")
print(f"PDF Sources: {len(PDF_SOURCES)}")


HTML Sources: 116
PDF Sources: 109


In [4]:
import os, json, time, requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import fitz  # PyMuPDF
from tqdm import tqdm

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
}

manifest_path = os.path.join(RAG_DIR, 'manifest.json')
manifest = {}
if os.path.exists(manifest_path):
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)

def save_manifest():
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

def get_with_retry(url, max_retries=3, timeout=20):
    """GET request with exponential backoff and SSL fallback."""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=timeout, verify=True)
            resp.raise_for_status()
            return resp
        except requests.exceptions.SSLError:
            try:
                resp = requests.get(url, headers=HEADERS, timeout=timeout, verify=False)
                resp.raise_for_status()
                return resp
            except Exception as e:
                if attempt == max_retries - 1:
                    raise e
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            wait = 2 ** attempt
            time.sleep(wait)
    return None

def harvest_pdfs_from_page(url, base_domain=None):
    """
    Visit a webpage and auto-discover all PDF links embedded inside it.
    Returns a list of absolute PDF URLs found on the page.
    """
    discovered = []
    try:
        resp = get_with_retry(url, timeout=15)
        if resp is None:
            return discovered
        soup = BeautifulSoup(resp.content, 'html.parser')
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href'].strip()
            # Check if link ends with .pdf or contains /pdf/ in path
            if href.lower().endswith('.pdf') or '/pdf/' in href.lower():
                # Convert relative URLs to absolute
                if href.startswith('http'):
                    abs_url = href
                elif href.startswith('//'):
                    abs_url = 'https:' + href
                elif href.startswith('/'):
                    parsed = urlparse(url)
                    abs_url = f"{parsed.scheme}://{parsed.netloc}{href}"
                else:
                    abs_url = urljoin(url, href)
                # Clean URL (remove fragments)
                abs_url = abs_url.split('#')[0]
                if abs_url not in discovered:
                    discovered.append(abs_url)
    except Exception as e:
        print(f"  ⚠️  PDF harvest failed for {url}: {e}")
    return discovered

def scrape_html(url):
    """Scrape HTML page text and save as .txt file."""
    if manifest.get(url) == 'success':
        return 'skipped'
    try:
        resp = get_with_retry(url)
        if resp is None:
            manifest[url] = 'failed: no response'
            return 'failed'
        soup = BeautifulSoup(resp.content, 'html.parser')
        # Remove script and style elements
        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()
        text = soup.get_text(separator=' ', strip=True)
        if len(text.strip()) < 100:
            manifest[url] = 'failed: too short'
            return 'failed'
        safe_name = url.replace('https://', '').replace('http://', '').replace('/', '_').replace('?', '_')[:80]
        out_path = os.path.join(RAG_DIR, f'html_{safe_name}.txt')
        with open(out_path, 'w', encoding='utf-8', errors='ignore') as f:
            f.write(f'SOURCE: {url}\n\n{text}')
        manifest[url] = 'success'
        save_manifest()
        return 'success'
    except Exception as e:
        manifest[url] = f'failed: {str(e)[:80]}'
        save_manifest()
        return 'failed'

def scrape_pdf(url):
    """Download a PDF and extract all text pages."""
    if manifest.get(url) == 'success':
        return 'skipped'
    try:
        resp = get_with_retry(url, timeout=45)
        if resp is None:
            manifest[url] = 'failed: no response'
            return 'failed'
        # Check content type
        content_type = resp.headers.get('Content-Type', '')
        if 'pdf' not in content_type.lower() and not url.lower().endswith('.pdf'):
            manifest[url] = 'failed: not a pdf'
            return 'failed'
        doc = fitz.open(stream=resp.content, filetype='pdf')
        text = '\n'.join([page.get_text() for page in doc])
        doc.close()
        if len(text.strip()) < 50:
            manifest[url] = 'failed: empty pdf (scanned image)'
            return 'failed'
        safe_name = url.replace('https://', '').replace('http://', '').replace('/', '_').replace('?', '_')[:80]
        out_path = os.path.join(RAG_DIR, f'pdf_{safe_name}.txt')
        with open(out_path, 'w', encoding='utf-8', errors='ignore') as f:
            f.write(f'SOURCE: {url}\n\n{text}')
        manifest[url] = 'success'
        save_manifest()
        return 'success'
    except Exception as e:
        manifest[url] = f'failed: {str(e)[:80]}'
        save_manifest()
        return 'failed'

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1: Scrape all HTML pages AND auto-harvest PDF links from each page
# ─────────────────────────────────────────────────────────────────────────────
print('=' * 60)
print('PHASE 1: Scraping HTML pages + Auto-harvesting embedded PDFs')
print('=' * 60)

discovered_pdfs = set()
html_stats = {'success': 0, 'skipped': 0, 'failed': 0}

for url in tqdm(HTML_SOURCES, desc='🌐 HTML Pages'):
    result = scrape_html(url)
    html_stats[result] = html_stats.get(result, 0) + 1

    # Auto-harvest PDF links from every HTML page
    if result in ('success', 'skipped'):
        pdfs_found = harvest_pdfs_from_page(url)
        for pdf_url in pdfs_found:
            discovered_pdfs.add(pdf_url)
        if pdfs_found:
            print(f'  📎 Found {len(pdfs_found)} PDFs on {url[:60]}')
    time.sleep(1)  # Polite delay

print(f'\n✅ HTML done: {html_stats}')
print(f'📎 Auto-discovered {len(discovered_pdfs)} PDFs from HTML pages')

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: Merge discovered PDFs with static PDF_SOURCES list
# ─────────────────────────────────────────────────────────────────────────────
all_pdf_urls = list(set(PDF_SOURCES) | discovered_pdfs)
print(f'\n📚 Total unique PDFs to process: {len(all_pdf_urls)}')
print(f'  → From static list: {len(PDF_SOURCES)}')
print(f'  → Auto-discovered:  {len(discovered_pdfs)}')

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3: Download & extract all PDFs
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('PHASE 3: Downloading & Extracting All PDFs')
print('=' * 60)

pdf_stats = {'success': 0, 'skipped': 0, 'failed': 0}
for url in tqdm(all_pdf_urls, desc='📄 PDFs'):
    result = scrape_pdf(url)
    pdf_stats[result] = pdf_stats.get(result, 0) + 1
    if result == 'success':
        time.sleep(0.5)  # Polite delay for successful downloads

print(f'\n✅ PDF done: {pdf_stats}')

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
txt_files = [f for f in os.listdir(RAG_DIR) if f.endswith('.txt')]
print('\n' + '=' * 60)
print('📊 SCRAPING COMPLETE — SUMMARY')
print('=' * 60)
print(f'  HTML pages scraped:      {html_stats["success"]} success, {html_stats["skipped"]} skipped, {html_stats["failed"]} failed')
print(f'  PDFs auto-discovered:    {len(discovered_pdfs)}')
print(f'  PDFs downloaded:         {pdf_stats["success"]} success, {pdf_stats["skipped"]} skipped, {pdf_stats["failed"]} failed')
print(f'  Total .txt files in RAG: {len(txt_files)}')
print(f'  Manifest entries:        {len(manifest)}')
print('=' * 60)


PHASE 1: Scraping HTML pages + Auto-harvesting embedded PDFs


🌐 HTML Pages:  28%|██▊       | 32/116 [03:28<03:14,  2.31s/it]

  📎 Found 9 PDFs on https://www.osha.gov/workers


🌐 HTML Pages:  28%|██▊       | 33/116 [03:29<02:43,  1.97s/it]

  📎 Found 3 PDFs on https://www.osha.gov/employers


...

  📎 Found 3 PDFs on https://www.euromonitor.com/consumer-foodservice


🌐 HTML Pages: 100%|██████████| 116/116 [12:04<00:00,  6.25s/it]



✅ HTML done: {'success': 0, 'skipped': 60, 'failed': 56}
📎 Auto-discovered 435 PDFs from HTML pages

📚 Total unique PDFs to process: 543
  → From static list: 109
  → Auto-discovered:  435

PHASE 3: Downloading & Extracting All PDFs


📄 PDFs: 100%|██████████| 543/543 [26:53<00:00,  2.97s/it]


✅ PDF done: {'success': 0, 'skipped': 277, 'failed': 266}

📊 SCRAPING COMPLETE — SUMMARY
  HTML pages scraped:      0 success, 60 skipped, 56 failed
  PDFs auto-discovered:    435
  PDFs downloaded:         0 success, 277 skipped, 266 failed
  Total .txt files in RAG: 338
  Manifest entries:        662


In [5]:
   from google.colab import files as colab_files

def ingest_local_pdf(local_path, source_label=None):
    """Extract text from a local PDF file and save it into RAG_DIR."""
    source_label = source_label or os.path.basename(local_path)
    try:
        doc = fitz.open(local_path)
        text = '\n'.join([page.get_text() for page in doc])
        doc.close()
        if len(text.strip()) < 50:
            print(f'  ⚠️  Skipped (empty / scanned image): {source_label}')
            return 'failed'
        safe_name = source_label.replace('/', '_').replace(' ', '_')[:80]
        out_path = os.path.join(RAG_DIR, f'custom_{safe_name}.txt')
        with open(out_path, 'w', encoding='utf-8', errors='ignore') as f:
            f.write(f'SOURCE: {source_label}\n\n{text}')
        print(f'  ✅ Ingested: {source_label}')
        return 'success'
    except Exception as e:
        print(f'  ❌ Failed: {source_label} ({e})')
        return 'failed'

custom_stats = {'success': 0, 'failed': 0}

# --- Option 1: interactive upload (optional — press cancel/skip to bypass) ---
print('📤 Optional: select PDFs to upload from your computer (or skip this dialog).')
try:
    uploaded = colab_files.upload()
    for fname in uploaded:
        result = ingest_local_pdf(fname, source_label=fname)
        custom_stats[result] = custom_stats.get(result, 0) + 1
except Exception as e:
    print(f'  (Upload skipped or unavailable: {e})')

# --- Option 2: scan the Drive drop folder ---
drive_pdfs = [f for f in os.listdir(CUSTOM_PDF_DIR) if f.lower().endswith('.pdf')]
print(f'\n📁 Found {len(drive_pdfs)} PDF(s) in {CUSTOM_PDF_DIR}')
for fname in tqdm(drive_pdfs, desc='📄 Custom Drive PDFs'):
    full_path = os.path.join(CUSTOM_PDF_DIR, fname)
    result = ingest_local_pdf(full_path, source_label=fname)
    custom_stats[result] = custom_stats.get(result, 0) + 1

print(f'\n✅ Custom PDF ingestion done: {custom_stats}')


📤 Optional: select PDFs to upload from your computer (or skip this dialog).



📁 Found 0 PDF(s) in /content/drive/MyDrive/FranchiseOps_AI/custom_pdfs


📄 Custom Drive PDFs: 0it [00:00, ?it/s]


✅ Custom PDF ingestion done: {'success': 0, 'failed': 0}


In [6]:
print(f"\nLoading {len(txt_files)} scraped text files from Drive...")
documents = []
for fname in tqdm(txt_files, desc="📂 Loading Docs"):
    filepath = os.path.join(RAG_DIR, fname)
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    if len(text.strip()) > 50:
        documents.append(Document(page_content=text, metadata={"source": fname, "type": "scraped"}))

print(f"\n✅ Loaded {len(documents)} documents into memory.")



Loading 338 scraped text files from Drive...


📂 Loading Docs: 100%|██████████| 338/338 [01:04<00:00,  5.21it/s]


✅ Loaded 338 documents into memory.


In [7]:

curated_sops = [
    {"id": "KB-101", "content": "Minimum freezer temperature must be maintained at -18°C or below at all times."},
    {"id": "KB-102", "content": "Store closing procedures include counting the cash drawer, securing the safe, turning off non-essential equipment, and setting the alarm."},
    {"id": "KB-103", "content": "Staff must wash hands with soap and warm water for at least 20 seconds before starting a shift, after using the restroom, and after handling raw meat."},
    {"id": "KB-104", "content": "Food preparation surfaces must be sanitized every 2 hours using the approved quaternary ammonium sanitizer solution."},
    {"id": "KB-105", "content": "The FIFO (First In, First Out) method must be strictly followed for all perishable inventory."},
    {"id": "KB-106", "content": "Daily temperature logs for all refrigeration units must be recorded at 8:00 AM, 2:00 PM, and 8:00 PM."},
    {"id": "KB-107", "content": "Customer complaints regarding food quality must be immediately escalated to the Shift Manager for resolution."},
    {"id": "KB-108", "content": "Spills on the customer floor must be marked with a wet floor sign and cleaned up within 3 minutes."},
    {"id": "KB-109", "content": "All staff members must wear the complete, approved uniform including name tag, hat/visor, and slip-resistant shoes."},
    {"id": "KB-110", "content": "Waste bins must be emptied when they are 3/4 full; never allow trash to overflow."},
    {"id": "KB-111", "content": "A minimum of 3 staff members (1 Manager, 1 Front-of-House, 1 Back-of-House) are required per shift."},
    {"id": "KB-112", "content": "FSSAI license must be prominently displayed near the point of sale at all times."},
    {"id": "KB-113", "content": "Penalties for FSSAI non-compliance can range from warning letters to license suspension and fines up to ₹2,00,000 depending on the severity."},
    {"id": "KB-114", "content": "Pest control services must be scheduled monthly, and inspection reports must be kept in the compliance binder."},
    {"id": "KB-115", "content": "Only approved vendors may be used for sourcing raw ingredients and packaging materials."},
    {"id": "KB-116", "content": "Deep fryers must be filtered daily and the oil completely changed every 3 days or when it fails the color check test."},
    {"id": "KB-117", "content": "Fire extinguishers must be inspected monthly by the Manager and annually by a certified professional."},
    {"id": "KB-118", "content": "All new employees must complete the 40-hour basic operational training program before working independently."},
    {"id": "KB-119", "content": "Cash drops to the safe must be performed whenever the register drawer exceeds ₹20,000."},
    {"id": "KB-120", "content": "The store key must never be duplicated, and must be returned immediately upon termination of employment."},
    {"id": "KB-121", "content": "Marketing ROI minimum threshold: >15% ROI is required for any marketing campaign renewal."},
    {"id": "KB-122", "content": "Weekly inventory counts must be reconciled against POS sales data, with variances above 2% investigated within 24 hours."},
    {"id": "KB-123", "content": "All POS reconciliation discrepancies over ₹500 must be reported to the Area Manager the same business day."},
    {"id": "KB-124", "content": "Social media responses to customer complaints must be posted within 2 hours during business hours, following the approved brand voice guide."},
    {"id": "KB-125", "content": "Customer refunds above ₹1,000 require Shift Manager approval and must be logged in the refund register with reason code."},
    {"id": "KB-126", "content": "All menu items containing the 8 major allergens (milk, egg, peanut, tree nuts, soy, wheat, fish, shellfish) must be clearly labeled on the menu board."},
    {"id": "KB-127", "content": "Franchise royalty payments are due by the 5th business day of each month, calculated as a percentage of gross monthly sales as per the franchise agreement."},
    {"id": "KB-128", "content": "All food-handling staff must renew their FOSTAC (Food Safety Training and Certification) certificate every 3 years."},
    {"id": "KB-129", "content": "In case of fire, staff must follow the evacuation route posted near each exit and assemble at the designated muster point."},
    {"id": "KB-130", "content": "Customer or employee personal data collected via loyalty programs must not be shared with third parties without documented consent."},
    {"id": "KB-131", "content": "Supplier audits must be conducted at least twice a year, covering hygiene, storage conditions, and documentation compliance."},
    {"id": "KB-132", "content": "Quality control checklists must be completed at the start and end of every shift and retained for a minimum of 90 days."},
    {"id": "KB-133", "content": "New employee onboarding includes a documented orientation covering safety protocols, POS training, and brand standards within the first 3 days."},
    {"id": "KB-134", "content": "Performance reviews for all store staff must be conducted quarterly, with documented goals and improvement plans where needed."},
    {"id": "KB-135", "content": "Disciplinary actions follow a progressive model: verbal warning, written warning, final warning, then termination, with HR sign-off required at each stage."},
    {"id": "KB-136", "content": "Employees under 18 must not work more than 4 consecutive hours without a 30-minute break, per applicable labour law."},
    {"id": "KB-137", "content": "Minimum wage rates vary by state/region and must be reviewed quarterly against the applicable government notification."},
    {"id": "KB-138", "content": "During POS software downtime, staff must switch to the manual order-taking backup process and reconcile all manual tickets once the system is restored."},
    {"id": "KB-139", "content": "Cash handling requires two-person verification for any cash drop exceeding ₹10,000."},
    {"id": "KB-140", "content": "Gift card balances must be verified against the central ledger before processing redemption of amounts over ₹2,000."},
    {"id": "KB-141", "content": "Loyalty program points expire 12 months after the date of issue unless otherwise stated in the current promotional terms."},
    {"id": "KB-142", "content": "Delivery aggregator commission reconciliation must be performed weekly, comparing aggregator payout reports against internal order logs."},
    {"id": "KB-143", "content": "Franchise agreement renewal discussions must begin at least 180 days before the current term's expiration date."},
    {"id": "KB-144", "content": "Non-compete clauses in the franchise agreement typically restrict a former franchisee from operating a competing outlet within a defined radius for 12–24 months."},
    {"id": "KB-145", "content": "Brand standards audits are scored out of 100, with a passing threshold of 85; scores below 70 trigger a mandatory corrective action plan."},
    {"id": "KB-146", "content": "Prior to a scheduled health inspection, managers must complete the pre-inspection checklist covering storage temperatures, pest control logs, and staff hygiene records."},
    {"id": "KB-147", "content": "Fire drills must be conducted at least twice a year, with attendance and timing documented in the safety log."},
    {"id": "KB-148", "content": "An updated emergency contact list, including local fire, police, and poison control numbers, must be posted near the manager's office."},
    {"id": "KB-149", "content": "First aid kits must be fully stocked and checked monthly; expired items must be replaced immediately."},
    {"id": "KB-150", "content": "Back-of-house areas must display clear signage for wet floors, hot surfaces, and sharp equipment at all times."},
    {"id": "KB-151", "content": "Any glass breakage in food-prep areas requires an immediate stop to service in that zone, full cleanup per the glass-breakage protocol, and disposal of all potentially contaminated food."},
    {"id": "KB-152", "content": "Cross-contamination prevention requires separate cutting boards and utensils color-coded for raw meat, poultry, seafood, and produce."},
    {"id": "KB-153", "content": "Cold-chain temperature abuse (any perishable item above 5°C for more than 2 hours) requires immediate disposal and incident log entry."},
    {"id": "KB-154", "content": "Water quality testing for ice machines and drinking water must be conducted quarterly by an approved lab."},
    {"id": "KB-155", "content": "Waste segregation into wet, dry, and recyclable categories is mandatory, with disposal logs retained for municipal compliance checks."},
    {"id": "KB-156", "content": "Energy and utility costs should be benchmarked monthly against the prior 3-month average; deviations above 15% require a maintenance inspection."},
    {"id": "KB-157", "content": "Local municipal trade and health licenses must be renewed at least 30 days before expiry to avoid operational disruption."},
    {"id": "KB-158", "content": "CCTV footage must be retained for a minimum of 30 days and made available to franchisor auditors upon request."},
    {"id": "KB-159", "content": "Employee grievances must be acknowledged within 48 hours and resolved or escalated within 2 weeks per the grievance redressal policy."},
    {"id": "KB-160", "content": "Workplaces with 10 or more employees in India must constitute an Internal Committee under the POSH Act to address complaints of sexual harassment."},
    {"id": "KB-161", "content": "In a social media crisis (viral complaint, health incident, etc.), the designated Communications Lead must be notified within 30 minutes and no employee should respond publicly without sign-off."},
    {"id": "KB-162", "content": "Vendor invoices must be matched against purchase orders and goods-received notes before payment approval (3-way match)."},
    {"id": "KB-163", "content": "Store managers must complete a documented handover checklist at every shift change, covering cash, inventory exceptions, and pending customer issues."},
    {"id": "KB-164", "content": "Any workplace injury must be logged in the incident register within 24 hours and reported to the Area Manager and, where required, the labour department."},
    {"id": "KB-165", "content": "New menu items require a documented allergen and nutritional review, plus a minimum 2-week staff training period before launch."},
]


In [8]:
# Convert curated SOP entries into Documents and add sentiment-tagged notes
for sop in curated_sops:
    documents.append(Document(
        page_content=sop["content"],
        metadata={"source": "curated_sops", "id": sop["id"], "type": "sop"}
    ))

print(f"✅ Added {len(curated_sops)} curated SOP entries.")
print(f"📚 Total documents now in memory: {len(documents)}")


✅ Added 65 curated SOP entries.
📚 Total documents now in memory: 403


In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore.save_local("franchiseops_faiss_index")
print(f"FAISS index built from {len(docs)} chunks and saved successfully to ./franchiseops_faiss_index")

# Optional: also persist the index to Drive so it survives across Colab sessions
DRIVE_INDEX_DIR = '/content/drive/MyDrive/FranchiseOps_AI/faiss_index'
vectorstore.save_local(DRIVE_INDEX_DIR)
print(f"💾 Also saved a copy to Drive: {DRIVE_INDEX_DIR}")


/tmp/ipykernel_630/2560372886.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index built from 18231 chunks and saved successfully to ./franchiseops_faiss_index
💾 Also saved a copy to Drive: /content/drive/MyDrive/FranchiseOps_AI/faiss_index


In [10]:
print("\n--- DRY RUN: Test Queries ---")
test_queries = [
    "What is the minimum freezer temperature?",
    "How many staff are required per shift?",
    "What is the handwashing procedure?",
    "What are the penalties for FSSAI non-compliance?",
    "How should customer complaints be escalated?",
    "What is the minimum marketing ROI threshold for campaign renewal?",
    "What are the staff performance review requirements?",
    "How often should supplier audits be conducted?",
    "What happens during a POS system outage?",
    "What is the process for handling a viral social media complaint?",
    "How long must CCTV footage be retained?",
    "What is required for POSH Act compliance in India?",
    "What is the protocol for glass breakage near food prep areas?",
    "How far in advance should franchise renewal talks start?",
]

for query in test_queries:
    print(f"\nQuery: {query}")
    results = vectorstore.similarity_search(query, k=1)
    if results:
        print(f"Answer (Snippet): {results[0].page_content}")
        print(f"Source: {results[0].metadata.get('source', 'Unknown')} | ID: {results[0].metadata.get('id', 'N/A')}")
    else:
        print("No relevant documents found.")



--- DRY RUN: Test Queries ---

Query: What is the minimum freezer temperature?
Answer (Snippet): Minimum freezer temperature must be maintained at -18°C or below at all times.
Source: curated_sops | ID: KB-101

Query: How many staff are required per shift?
Answer (Snippet): A minimum of 3 staff members (1 Manager, 1 Front-of-House, 1 Back-of-House) are required per shift.
Source: curated_sops | ID: KB-111

Query: What is the handwashing procedure?
Answer (Snippet): 


Ensure handwashing stations with potable 
water, soap, and a method to dry hands are 
available to workers. Additionally, ensure hand 
sanitizer with at least 60% alcohol is available.



Ensure potable water is provided for 
drinking, personal hygiene, cooking, washing 
of goods, washing of utensils, washing of 
food preparation or processing premises, 
and rooms not directly connected with the 
production or service performed by the 
establishment (e. g. , first-aid, medical services, 
and dressing).



Source

In [ ]:

while True:
    query = input("\n🔎 Ask a question (or type 'exit'): ").strip()
    if not query or query.lower() in ("exit", "quit"):
        print("Done.")
        break

    results = vectorstore.similarity_search(query, k=3)
    if not results:
        print("No relevant documents found.")
        continue

    for rank, r in enumerate(results, start=1):
        print(f"\n--- Match {rank} ---")
        print(r.page_content[:800])
        print(f"(source: {r.metadata.get('source', 'unknown')} | id: {r.metadata.get('id', 'N/A')})")



🔎 Ask a question (or type 'exit'):  What happens during a POS system outage?

--- Match 1 ---
During POS software downtime, staff must switch to the manual order-taking backup process and reconcile all manual tickets once the system is restored.
(source: curated_sops | id: KB-138)

--- Match 2 ---
provided for any reasonably foreseeable 
repair emergency.



When machine operations, configuration, 
or size make it necessary for the operator 
to leave the control station, and part of the 
machine could move if accidentally activated, 
the part is separately locked out or blocked.



Additional Resources
•	 OSHA Regulations: 29 CFR 1910.147, Control of Hazardous Energy (Lockout/Tagout)
•	 OSHA: Control of Hazardous Energy (Lockout/Tagout)
•	 OSHA: Lockout-Tagout Interactive Training Program
•	 NIOSH: Using Lockout and Tagout Procedures to Prevent Injury and Death during 
Machine Maintenance 
Reset Form
(source: pdf_www.osha.gov_sites_default_files_publications_small-business.pdf.t